In [4]:
import requests
import os
import pandas as pd
from tqdm import tqdm
from datetime import date
from bs4 import BeautifulSoup, Comment
from copy import deepcopy

In [65]:
def clean_columns(df):
    df.columns = df.columns.get_level_values(1)
    dupIndex = list(df.columns).index('W-L%',list(df.columns).index('W-L%')+1)
    df.columns.values[dupIndex] = 'W-L% plyf'
    return df

In [10]:
# TODO: account for active coach or not
def get_coach(coach,fetch=False):
    url = f'https://www.pro-football-reference.com/coaches/{coach}.htm'
    file_name = f"input/coaches/tables/{url.split('/')[-1].split('.')[0]}.csv"

    if os.path.exists(file_name):
        return pd.read_csv(file_name,index_col=0)

    else :
        ret = requests.get(url).text
        ret = pd.read_html(ret)[0]
        ret = clean_columns(ret)
        ret.to_csv(file_name)
        return ret

In [ ]:
# TODO: reading in every coach from 2023
coaches = pd.read_csv('input/coaches23.csv')
for c in tqdm(coaches['name'].values):
    get_coach(c)

100%|██████████| 524/524 [00:00<00:00, 1330.72it/s]


In [12]:
get_coach(coaches.iloc[0][0]).head()

,Year,Age,Tm,Lg,G,W,L,T,W-L%,SRS,OSRS,DSRS,G plyf,W plyf,L plyf,W-L% plyf,Rank,Notes
0,1963,33,BAL,NFL,14,8,6,0,0.571,2.0,0.8,1.2,NaN,NaN,NaN,NaN,3.0,NaN
1,1964,34,BAL,NFL,14,12,2,0,0.857,15.5,9.3,6.2,1.0,0.0,1.0,0.0,1.0,NaN
2,1965,35,BAL,NFL,14,10,3,1,0.769,12.9,7.0,5.9,1.0,0.0,1.0,0.0,1.0,NaN
3,1966,36,BAL,NFL,14,9,5,0,0.643,7.6,2.2,5.5,NaN,NaN,NaN,NaN,2.0,NaN
4,1967,37,BAL,NFL,14,11,1,2,0.917,13.2,6.4,6.8,NaN,NaN,NaN,NaN,1.0,NaN


### TODO
1. convert and save processed tables -> cut bottom rows from tables and label, likely trim Lg, G, W, L, T, Rank, W plyf, DSRS
--> toy with whether certain columns matter -> make this dynamic
2. run quick and dirty model on just that season, w/o any knowledge of other seasons --> call it "online"
3. add exp columns to rows, then see if model outperforms

In [9]:
def fixCol(df,col,typ):
    df[col] = df[col].fillna(0).astype(typ)
    return df

In [145]:
def clean_rows(coach,footerBool=False):
    df = get_coach(coach)
    footer = df[~(df['Year'].str.isdigit())]
    rows = df[df['Year'].str.isdigit()].copy()

    if 'Num' not in df.columns :
        df.insert(len(df.columns)-1,'Num',0)
        df.insert(len(df.columns)-1,'Won',0)
    
    rows = rows.fillna(0)
    
    for col in ['Age','Year']:
        rows[col] = rows[col].astype(int)

    return rows if not footerBool else footer

In [129]:
def label_fired(coach):
    df = clean_rows(coach)
    df['Fired'] = 0
    last_years = df.groupby('Tm')['Year'].idxmax()
    for idx in last_years:
        if df.loc[idx, 'Year'] != 2023:
            df.loc[idx, 'Fired'] = 1    
    return df

### Quick and Dirty Online Model

Fix ConzJi0's double 1922 head coaching year and GibsGe0's double 1930 head coaching year

In [137]:
shula = coaches.iloc[0][0]
coach2 = coaches.iloc[0][0]
label_fired(shula).tail()

,Year,Age,Tm,Lg,G,W,L,T,W-L%,SRS,OSRS,DSRS,G plyf,W plyf,L plyf,W-L% plyf,Rank,Notes,Fired
28,1991,61,MIA,NFL,16,8,8,0,0.500,-4.1,0.7,-4.8,0.0,0.0,0.0,0.0,3.0,0,0
29,1992,62,MIA,NFL,16,11,5,0,0.688,1.5,2.1,-0.6,2.0,1.0,1.0,0.5,1.0,0,0
30,1993,63,MIA,NFL,16,9,7,0,0.563,-0.5,3.9,-4.3,0.0,0.0,0.0,0.0,2.0,0,0
31,1994,64,MIA,NFL,16,10,6,0,0.625,4.2,4.5,-0.3,2.0,1.0,1.0,0.5,1.0,0,0
32,1995,65,MIA,NFL,16,9,7,0,0.563,2.7,3.1,-0.4,1.0,0.0,1.0,0.0,3.0,0,1


In [153]:
def quick_dirty(coach):
    df = label_fired(coach)
    return df[['W-L%','SRS','W-L% plyf','Fired']]

In [143]:
def aggregate(method):
    coaches = pd.read_csv('input/coaches23.csv')['name'].values
    dfs = [method(coach) for coach in coaches]
    return pd.concat(dfs).reset_index(drop=True)

#### Model

In [157]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [158]:
data = aggregate(quick_dirty)
X = data.drop(columns=['Fired'])
y = data['Fired']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(classification_report(y_test, y_pred))


Accuracy: 0.7450592885375494
              precision    recall  f1-score   support

           0       0.77      0.92      0.84       360
           1       0.61      0.32      0.42       146

    accuracy                           0.75       506
   macro avg       0.69      0.62      0.63       506
weighted avg       0.72      0.75      0.72       506

